In [ ]:
import pandas as pd
from pathlib import Path

processed_books = pd.read_csv(r"../data/processed_books_dataset.csv")
folder = Path(r"../data/bestsellers")
csv_files = list(folder.glob("*.csv"))

bestseller_dfs = []

for file in csv_files:
    df = pd.read_csv(file)

    # Preserve which table the book came from
    df["bestseller_source"] = file.stem

    bestseller_dfs.append(df)

bestsellers_df = pd.concat(
    bestseller_dfs,
    ignore_index=True
)


print("Total records:", len(bestsellers_df))
print(bestsellers_df.head())


In [ ]:
import pandas as pd
import re
import unicodedata
from rapidfuzz import fuzz

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()

    # Remove punctuation except commas, which separate authors
    text = re.sub(r"[^\w\s,]", "", text, flags=re.UNICODE)

    # Normalize spaces without removing author separators
    text = re.sub(r"\s+", " ", text).strip()

    # Clean spaces around commas
    text = re.sub(r"\s*,\s*", ", ", text)

    return text

In [ ]:
processed_books["title_key"] = processed_books["title"].apply(normalize_text)
bestsellers_df["title_key"] = bestsellers_df["Book"].apply(normalize_text)

processed_books["author_key"] = processed_books["authors"].apply(normalize_text)
bestsellers_df["author_key"] = bestsellers_df["Author(s)"].apply(normalize_text)

In [ ]:
import pandas as pd
from rapidfuzz import fuzz

# Create a unique list of bestseller title-author pairs
bestseller_keys = (
    bestsellers_df[["title_key", "author_key"]]
    .drop_duplicates()
    .dropna(subset=["title_key"])
)

# Prepare columns in the main dataset
processed_books["is_bestseller"] = False
processed_books["bestseller_match_method"] = None

# Group bestseller records by title for faster candidate lookup
bestsellers_by_title = {
    title: group
    for title, group in bestseller_keys.groupby("title_key")
}

for idx, book in processed_books.iterrows():

    title = book["title_key"]
    author = book["author_key"]

    # Find bestseller records with the same normalized title
    candidates = bestsellers_by_title.get(title)

    if candidates is None:
        continue

    # Compare this book's author against each candidate author
    best_score = 0

    for _, candidate in candidates.iterrows():
        candidate_author = candidate["author_key"]

        if pd.isna(author) or pd.isna(candidate_author):
            continue

        score = fuzz.ratio(
            str(author),
            str(candidate_author)
        )

        best_score = max(best_score, score)

    # Use a conservative starting threshold
    if best_score >= 90:
        processed_books.at[idx, "is_bestseller"] = True
        processed_books.at[
            idx, "bestseller_match_method"
        ] = "exact_title_fuzzy_author"

In [ ]:
processed_books[processed_books["is_bestseller"] == True].head(20)

In [ ]:
processed_books[processed_books["is_bestseller"]==True].count()

In [ ]:
import sqlite3

DB_PATH = r"../data/books.db"

# Connect to the database (creates it if it doesn't exist)
conn = sqlite3.connect(DB_PATH)

# Save the DataFrame into a table called processed_books
processed_books.to_sql(
    "processed_books",
    conn,
    if_exists="replace",
    index=False
)

conn.close()


In [ ]:
import sqlite3

def get_connection():
    return sqlite3.connect(DB_PATH)


def create_tables():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS processed_books (
            google_id TEXT PRIMARY KEY,

            title TEXT,
            subtitle TEXT,
            authors TEXT,
            publisher TEXT,
            published_date TEXT,
            description TEXT,
            categories TEXT,
            page_count REAL,
            language TEXT,

            isbn_10 TEXT,
            isbn_13 TEXT,
            thumbnail TEXT,

            openlibrary_categories TEXT,

            title_clean TEXT,
            author_clean TEXT,
            category_clean TEXT,
            clean_description TEXT,
            author_set TEXT,

            completeness REAL
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS awards (
            award_id INTEGER PRIMARY KEY AUTOINCREMENT,
            award_name TEXT UNIQUE
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS book_awards (
            google_id TEXT,
            award_id INTEGER,

            PRIMARY KEY (google_id, award_id),

            FOREIGN KEY (google_id)
                REFERENCES books(google_id),

            FOREIGN KEY (award_id)
                REFERENCES awards(award_id)
        )
    """)

    conn.commit()
    conn.close()


if __name__ == "__main__":
    create_tables()
    print("Database created successfully.")

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = "../data/books.db"
CSV_PATH = r"../data/processed_books_dataset.csv"


BOOK_COLUMNS = [
    "google_id",
    "title",
    "subtitle",
    "authors",
    "publisher",
    "published_date",
    "description",
    "categories",
    "page_count",
    "language",
    "isbn_10",
    "isbn_13",
    "thumbnail",
    "openlibrary_categories",
    "title_clean",
    "author_clean",
    "category_clean",
    "clean_description",
    "author_set",
    "completeness"
]


def import_books():
    df = pd.read_csv(CSV_PATH)

    books = df[BOOK_COLUMNS].copy()

    books = books.fillna("")

    books = books.drop_duplicates(
        subset="google_id"
    )

    conn = sqlite3.connect(DB_PATH)

    books.to_sql(
        "books",
        conn,
        if_exists="append",
        index=False
    )

    conn.close()

    print(f"Imported {len(books)} books.")


if __name__ == "__main__":
    import_books()

In [ ]:
conn = sqlite3.connect("../data/books.db")

conn.execute("DELETE FROM book_awards")
conn.execute("DELETE FROM awards")

conn.commit()
conn.close()

print("Award tables cleared.")

In [ ]:
import sqlite3
import pandas as pd
import re
from rapidfuzz.fuzz import ratio

# --------------------------------
# 1. Load CSV files
# --------------------------------

AWARDS_CSV = r"../data/books_by_award.csv"
CSV_PATH = r"../data/processed_books_dataset.csv"

awards_df = pd.read_csv(AWARDS_CSV)

print("Award records:", len(awards_df))

# --------------------------------
# 2. Load books from SQLite
# --------------------------------

conn = sqlite3.connect("../data/books.db")

books = pd.read_sql_query("""
    SELECT
        google_id,
        title,
        authors,
        title_clean,
        author_clean
    FROM books
""", conn)

conn.close()

print("Books in database:", len(books))

# --------------------------------
# 3. Normalize award titles
# --------------------------------

def normalize_award_title(title):
    if pd.isna(title):
        return ""

    title = str(title).strip()

    # Remove a trailing Wikipedia disambiguation label,
    # such as "(novel)" or "(Enright novel)".
    title = re.sub(r"\s*\([^)]*\)\s*$", "", title)

    title = title.lower()
    title = title.replace("&", " and ")
    title = re.sub(r"[^\w\s]", " ", title)
    title = re.sub(r"\s+", " ", title)

    return title.strip()


awards_df["award_title_clean"] = (
    awards_df["book"].apply(normalize_award_title)
)

books["title"] = books["title"].fillna("").astype(str)

# Use the actual database title for matching.
books["database_title_clean"] = (
    books["title"].apply(normalize_award_title)
)

# --------------------------------
# 4. Classify title matches
# --------------------------------

def title_match_type(award_title, database_title):
    award_title = normalize_award_title(award_title)
    database_title = normalize_award_title(database_title)

    if not award_title or not database_title:
        return None, 0

    score = ratio(award_title, database_title)

    if award_title == database_title:
        return "exact", score

    # Avoid fuzzy matching short titles.
    if len(award_title.split()) < 3:
        return None, score

    if score >= 95:
        return "fuzzy", score

    return None, score


# --------------------------------
# 5. Generate potential matches
# --------------------------------

potential_matches = []

for _, award_row in awards_df.iterrows():
    award_title = award_row["book"]
    award_name = award_row["award"]

    for db_idx, book_row in books.iterrows():
        match_type, score = title_match_type(
            award_title,
            book_row["title"]
        )

        if match_type is not None:
            potential_matches.append({
                "award": award_name,
                "award_book": award_title,
                "database_title": book_row["title"],
                "google_id": book_row["google_id"],
                "authors": book_row["authors"],
                "match_type": match_type,
                "score": score
            })

matches_df = pd.DataFrame(potential_matches)

# --------------------------------
# 6. Display and save results
# --------------------------------

if matches_df.empty:
    print("No potential matches found.")

else:
    print("\nMatch counts by type:")
    print(matches_df["match_type"].value_counts())

    print("\nPotential matches:")
    print(
        matches_df.sort_values(
            ["match_type", "score"],
            ascending=[True, False]
        ).to_string(index=False)
    )

    print("\nAward books matching multiple database records:")
    duplicate_matches = (
        matches_df.groupby(["award", "award_book"])
        .agg(
            match_count=("google_id", "nunique"),
            database_titles=("database_title", " | ".join),
            authors=("authors", " | ".join)
        )
        .reset_index()
    )

    duplicate_matches = duplicate_matches[
        duplicate_matches["match_count"] > 1
    ]

    print(duplicate_matches.to_string(index=False))

    matches_df.to_csv(
        "award_potential_matches.csv",
        index=False
    )

    print("\nSaved to award_potential_matches.csv")

In [ ]:
# Mark each candidate for review.
matches_df["review_status"] = "review"

# Google Books IDs verified from your output.
keep_ids = {
    "ZMLkjwEACAAJ",  # A Brief History of Seven Killings
    "1lI_PgAACAAJ",  # The God of Small Things
    "v8qW0AEACAAJ",  # The Inheritance of Loss
    "6RS00wtlWhIC",   # Life of Pi â€” Yann Martel
    "X16yzwEACAAJ",  # The Luminaries
    "N1mwQgAACAAJ",  # The Remains of the Day
    "lqsgzwEACAAJ",  # The Sense of an Ending
    "zWPAGgAACAAJ",  # The Amber Spyglass
    "4ptXvQEACAAJ",  # Bridget Jones's Diary â€” Helen Fielding
    "A-AS46wG0p8C",  # The Da Vinci Code
    "G_xavgEACAAJ",  # Fifty Shades of Grey
    "OWIauAEACAAJ",  # Harry Potter and the Half-Blood Prince
    "xYotngEACAAJ",  # Harry Potter and the Philosopher's Stone
    "zjRQAAAAYAAJ",  # Northern Lights â€” Philip Pullman
    "j-6LMQEACAAJ",  # The Ocean at the End of the Lane
    "NbPjjwEACAAJ",  # A Series of Unfortunate Events (author missing)
    "0sBu1Fj4Ed0C",  # Wild Swans
}

matches_df.loc[
    matches_df["google_id"].isin(keep_ids),
    "review_status"
] = "candidate_keep"

matches_df.to_csv(
    "../data/award_matches.csv",
    index=False
)

print(matches_df["review_status"].value_counts())

In [ ]:
import sqlite3
import pandas as pd

VERIFIED_CSV = "../data/award_matches.csv"

# Load only manually verified relationships.
verified_df = pd.read_csv(VERIFIED_CSV)

required_columns = {"award", "google_id"}
if not required_columns.issubset(verified_df.columns):
    raise ValueError(
        "CSV must contain 'award' and 'google_id' columns."
    )

verified_df = verified_df.dropna(
    subset=["award", "google_id"]
).copy()

verified_df["award"] = verified_df["award"].astype(str).str.strip()
verified_df["google_id"] = (
    verified_df["google_id"].astype(str).str.strip()
)

verified_df = verified_df[
    (verified_df["award"] != "")
    & (verified_df["google_id"] != "")
]

# Remove repeated relationships in the CSV.
verified_df = verified_df.drop_duplicates(
    subset=["award", "google_id"]
)

conn = sqlite3.connect(DB_PATH)

try:
    conn.execute("PRAGMA foreign_keys = ON")

    cursor = conn.cursor()

    # Confirm both tables exist.
    cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name IN ('books', 'awards', 'book_awards')
    """)

    existing_tables = {row[0] for row in cursor.fetchall()}

    required_tables = {"books", "awards", "book_awards"}
    if not required_tables.issubset(existing_tables):
        raise RuntimeError(
            "Missing tables. Create books, awards, and book_awards first."
        )

    # Check that every Google Books ID exists in the books table.
    missing_ids = []

    for google_id in verified_df["google_id"].unique():
        cursor.execute(
            "SELECT 1 FROM books WHERE google_id = ?",
            (google_id,)
        )
        if cursor.fetchone() is None:
            missing_ids.append(google_id)

    if missing_ids:
        raise ValueError(
            f"These google_id values are missing from books: {missing_ids}"
        )

    # Insert each distinct award name once.
    award_names = verified_df["award"].unique()

    for award_name in award_names:
        cursor.execute("""
            INSERT OR IGNORE INTO awards (award_name)
            VALUES (?)
        """, (award_name,))

    # Link each verified book to its award.
    inserted_links = 0

    for _, row in verified_df.iterrows():
        cursor.execute(
            "SELECT award_id FROM awards WHERE award_name = ?",
            (row["award"],)
        )
        award_id = cursor.fetchone()[0]

        cursor.execute("""
            INSERT OR IGNORE INTO book_awards
                (google_id, award_id)
            VALUES (?, ?)
        """, (row["google_id"], award_id))

        inserted_links += cursor.rowcount

    conn.commit()

    print("Verified relationships processed:", len(verified_df))
    print("New book-award links inserted:", inserted_links)

    # Show the imported relationships.
    result = pd.read_sql_query("""
        SELECT
            b.title,
            b.authors,
            a.award_name
        FROM book_awards AS ba
        JOIN books AS b
            ON b.google_id = ba.google_id
        JOIN awards AS a
            ON a.award_id = ba.award_id
        ORDER BY a.award_name, b.title
    """, conn)

    print("\nCurrent award relationships:")
    print(result.to_string(index=False))

except Exception:
    conn.rollback()
    raise

finally:
    conn.close()

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

award_books = pd.read_sql_query("""
    SELECT
        ba.google_id,
        a.award_name,
        a.award_id
    FROM book_awards AS ba
    JOIN awards AS a
        ON ba.award_id = a.award_id
""", conn)

conn.close()

print(award_books.head(10))

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = r"../data/books.db"

conn = sqlite3.connect(DB_PATH)

award_books = pd.read_sql_query(
    """
    SELECT DISTINCT google_id, award_id
    FROM book_awards
    """,
    conn
)

conn.close()

conn = sqlite3.connect(DB_PATH)

award_books = pd.read_sql_query("""
    SELECT
        ba.google_id,
        a.award_name,
        a.award_id
    FROM book_awards AS ba
    JOIN awards AS a
        ON ba.award_id = a.award_id
""", conn)

conn.close()

print(award_books.head(10))

# Group award names by book
awards_by_book = (
    award_books
    .dropna(subset=["google_id", "award_name"])
    .groupby("google_id")["award_name"]
    .agg(lambda x: sorted(set(x)))
)

processed_books["award_names"] = (
    processed_books["google_id"]
    .map(awards_by_book)
    .apply(lambda x: x if isinstance(x, list) else [])
)

processed_books["is_award_winner"] = (
    processed_books["award_names"].str.len() > 0
)

print(
    processed_books.loc[
        processed_books["is_award_winner"],
        ["title", "google_id", "award_names"]
    ].to_string(index=False)
)

In [ ]:
conn = sqlite3.connect(DB_PATH)

print("Book-award links:")
print(pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM book_awards",
    conn
))

print("\nAwards:")
print(pd.read_sql_query(
    "SELECT * FROM awards",
    conn
))

print("\nSample book-award relationships:")
print(pd.read_sql_query("""
    SELECT b.google_id, b.title, a.award_name, a.award_id
    FROM book_awards AS ba
    JOIN books AS b ON b.google_id = ba.google_id
    JOIN awards AS a ON a.award_id = ba.award_id
    LIMIT 10
""", conn))

conn.close()

In [ ]:
processed_books.to_csv(
    "../data/processed_books_with_bestsellers.csv",
    index=False
)

In [ ]:
processed_books.head()